In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt


In [2]:
# Load the dataset
file_path = '../../data/Data3.csv'  # Make sure the file is in the same directory as the code
data = pd.read_csv(file_path)

# Select relevant columns
columns_to_normalize = ['AVG_NO_USER', 'AVG_USR_THRPUT_DL', 'DL_TRAFFIC_MB']
data = data[['EUTRANCELLFDD'] + columns_to_normalize]

# Normalize the data between 0-1 for LSTM
scaler = MinMaxScaler(feature_range=(0, 1))
data[columns_to_normalize] = scaler.fit_transform(data[columns_to_normalize])

# Group data by 'EUTRANCELLFDD'
grouped_data = data.groupby('EUTRANCELLFDD')

In [3]:
# Function to create LSTM model
def create_lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=input_shape))
    model.add(LSTM(50, return_sequences=False))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Function to create time-series sequences for LSTM
def create_sequences(data, time_steps):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:(i + time_steps), :-1])
        y.append(data[i + time_steps, -1])
    return np.array(X), np.array(y)

# To store MSE for each EUTRANCELLFDD
mse_results = {}

# Time steps for LSTM
time_steps = 3

In [4]:
# Process each EUTRANCELLFDD for 'DL_TRAFFIC_MB'
for cell, group in grouped_data:
    # Ensure there are enough samples to split
    if len(group) < 4:  # Minimum required for a reasonable split
        continue
    
    # Prepare data (using 'DL_TRAFFIC_MB' for LSTM univariate forecasting)
    data_values = group[['AVG_NO_USER', 'AVG_USR_THRPUT_DL', 'DL_TRAFFIC_MB']].values
    
    # Create sequences
    X, y = create_sequences(data_values, time_steps)
    
    # Split into train and test sets (80-20 split)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Create and train LSTM model
    model = create_lstm_model((X_train.shape[1], X_train.shape[2]))
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Predict on the test set
    y_pred = model.predict(X_test)
    
    # Calculate MSE
    test_mse = mean_squared_error(y_test, y_pred)
    
    # Store MSE results
    mse_results[cell] = {'test_mse': test_mse}


3/3 [==============================] - 1s 3ms/step


1/1 [==============================] - 2s 2s/step


3/3 [==============================] - 1s 2ms/step


3/3 [==============================] - 1s 3ms/step


3/3 [==============================] - 1s 2ms/step


ValueError: With n_samples=1, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.